# A1 malicious-fraction grid: 10%, 20%, 30%, 5 rounds each

LEFT.md §1.6's fraction-sweep grid, A1 (model-replacement scaling)
only -- see this notebook's generator script
(`notebooks/_build_attack_fraction_grid_notebook.py`) for why A3/A4-
replay/A5 are NOT in this grid (they attack the ring-signature/VRF/
key-image layer directly, not the trained delta, and have no real
per-round integration to sweep a fraction over; their isolated-test
results, already final, are what the paper cites for those three).

**Cells in this notebook, in run order:** 10% malicious (5 of 50 clients), 20% malicious (10 of 50 clients), 30% malicious (15 of 50 clients).
Same model/dataset as LEFT.md item #1 (`tiny_cnn` + MNIST, case1) --
this grid is about the per-round V3-rejection rate as a function of
malicious fraction, not a new accuracy result.

**Deliberately 5 rounds, not 20**: the claim this grid
supports -- "V3 rejects every eligible attacker, every round, at this
fraction" -- is decided on round 0 just as validly as round 19; a short
run gets the same answer at a fraction of the compute.

**Expect more ABORTED rounds at higher fractions** -- at `p=0.25,
n=50, k_min=8`, malicious clients are excluded from `k_min`-eligibility,
so predicted abort probability rises with fraction: ~9.4% at 10%,
~18.2% at 20%, ~32.2% at 30% (LEFT.md §1.6's own table). This is
the sampling model working as designed; the run cell below reports
aborted rounds separately from V3-rejection verdicts so the two are
never conflated into one number.

**Checkpoints**: each of the 3 cells gets its own dedicated git branch
(`colab-checkpoints-case1-tiny_cnn-fracgrid-<10pct|20pct|30pct>`) and its own
clone directory, so none can collide with each other or with any
other notebook in this repo.

**Runtime**: 5 rounds is a small fraction of a full attack7
run's ~5 hours -- expect well under an hour per cell, under 2 hours
total, including DKG/ring setup for each.


In [ ]:
# 1. Clone/pull the repo. (No Drive mount -- checkpoints go through
# git instead, see the git-checkpoint-setup cell further down and this
# notebook's own intro markdown for why.)
import os

if not os.path.exists('/content/FL-ZKP'):
    !git clone https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
else:
    !cd /content/FL-ZKP && git fetch -q origin main && git reset -q --hard origin/main
%cd /content/FL-ZKP


In [ ]:
# 2. Install the EC-dependent toolchain (rbcl for ristretto255, torch/torchvision, flwr-datasets).
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121


**If this is the first time cell 2 has installed anything in this
session, restart the kernel now: Runtime -> Restart session.** Then
continue from cell 3 below (NOT from cell 1 -- Drive stays mounted and
the repo stays cloned across a restart, only the Python process
resets, so re-running cell 1 would be harmless but pointless).


In [ ]:
# 3. Sanity-check the install.
import sys
sys.path.insert(0, '/content/FL-ZKP')
import rbcl
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "see crypto/ristretto.py's import fallback and FIX.md F8."
)
import torch
print("rbcl OK. torch OK, CUDA available:", torch.cuda.is_available())


In [ ]:
# 4. Load config.
import yaml
with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)


In [ ]:
# 5. Load MNIST, stratified-IID split across clients.
# NOTE: configs/mvp_3day.yaml's datasets.mnist.source is "csv" (reads a
# Drive-cached CSV that this run's Drive doesn't have) -- that path is
# for whoever already staged the CSV. This run instead forces
# source="torchvision", which downloads MNIST directly over the network,
# no Drive dependency at all. Deliberately NOT changed in the shared
# config file, since other runs may rely on the csv path being default.
from fl.data_iid import prepare_case1

case1_data = prepare_case1(seed=CFG['seed'], source="torchvision")
print("clients:", len(case1_data['client_train']))
print("server_root size:", len(case1_data['server_root'].labels))
print("global_val size:", len(case1_data['global_val'].labels))


## Grid cell 1 of 3: 10% malicious (5 of 50 clients)

In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, verify_sub_share, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]

# Feldman-verify every sub-share before finalizing -- a real committee
# would discard a contribution that fails this and re-run DKG without
# the cheating holder; all contributions here are honest, so every
# check is expected to pass. Same check notebooks/traceability_demo
# already runs standalone; added here (2026-11-19) alongside pk_shares
# below so this notebook's committee is held to the same standard, now
# that §6.2 opening runs for real from this notebook's own evidence
# (not just the standalone demo).
for c in dkg_contributions:
    for j in key_holder_ids:
        assert verify_sub_share(c.sub_shares[j], c.feldman_commitments), (
            f"holder {c.holder_id}'s sub-share to {j} failed Feldman verification"
        )

PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
# pk_shares (2026-11-19): each holder's PUBLIC Feldman commitment --
# needed by crypto.elgamal_threshold.verify_partial inside
# fl/trace.py::run_opening_protocol to check a partial decryption
# before combining it. key_holder_shares (above) are the holders' own
# SECRET shares, needed to CREATE a partial in the first place
# (crypto.elgamal_threshold.compute_partial) -- both come out of the
# SAME DKG run, never recomputed independently.
pk_shares = {i: pk_share_for(i, dkg_contributions) for i in key_holder_ids}
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")


In [ ]:
# 7. Build the model.
# ~20.6K params, same model/dataset as LEFT.md item #1 and every attack7 notebook's base case.
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=10, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


## Checkpoint durability via git (not Drive)

**Why a separate clone, a separate branch, and a callback -- not just
`git commit` inside `/content/FL-ZKP`:**
- A SEPARATE clone (`/content/FL-ZKP-checkpoints`) so committing a
  checkpoint can never touch the code checkout this notebook is actually
  importing from -- no risk of a `git checkout`/`reset` on the
  checkpoint side disturbing `sys.path`-imported modules mid-run.
- A dedicated branch (shown as `CKPT_BRANCH` in the cell below), never
  `main` -- small binary commits (one per round, re-run disconnects
  add more) have no business sitting in the same history a reviewer
  sees when they clone this repo to read the paper's actual code.
  **Each of the 4 MVP run notebooks gets its OWN branch** (FIX.md F17,
  2026-11-17) -- an earlier version shared one branch across all four,
  keyed apart only by checkpoint filename, and every single push
  collided when they ran concurrently (observed live: 12/12 rounds
  failed), even with a fetch+rebase retry (F16). A dedicated ref per
  notebook removes the race at the git level instead of mitigating it.
- `fl/round_loop.py`'s `on_checkpoint_saved` hook (2026-11-16) keeps
  the FL library itself git-agnostic -- it just calls back with
  `(checkpoint_path, round_idx)` after every local save; all the git
  mechanics below live here, in the notebook, not in the library.

**You'll need a GitHub Personal Access Token** with push access to
`YashasweeNarayan/FL-ZKP` (classic PAT, `repo` scope, or a fine-grained
token scoped to just this repo). The cell below prompts for it via
`getpass` (never echoed to the cell's output, never written to a file
or committed) and deletes it from memory immediately after use.

**On a disconnect and re-run**: this clone lives on local Colab disk,
so it does NOT survive the disconnect (same as the code clone) -- but
everything pushed to this notebook's own `CKPT_BRANCH` before the
disconnect DID survive, on GitHub. Re-running this cell re-clones
fresh and fetches that branch if it already exists, so the run cell's
`resume_from` still finds the latest checkpoint and continues from
there, not round 0.


In [ ]:
# 7.5. Git-backed checkpoint durability (replaces Drive -- see markdown above).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints-fracgrid-0"
# 2026-11-17: each of the 4 MVP run notebooks gets its OWN branch, not
# one shared branch -- see FIX.md F17. Running all four at once on one
# shared branch caused every single push to collide and fail (12/12
# rounds, observed live) even with F16's fetch+rebase retry, because
# the retry itself can lose the SAME race under real concurrent load.
# A dedicated ref per notebook removes the race at the git level
# instead of mitigating it after the fact.
CKPT_BRANCH = "colab-checkpoints-case1-tiny_cnn-fracgrid-10pct"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    # does this notebook's own CKPT_BRANCH already exist on the remote
    # (pushed in an earlier, since-disconnected session)? if so, use it --
    # that's where the actual latest checkpoint lives. If not, this is new.
    # explicit destination refspec -- a plain "fetch origin <branch>"
    # only updates FETCH_HEAD, not refs/remotes/origin/<branch>, so the
    # checkout below would fail even though the fetch itself succeeded.
    # Verified against a real throwaway git remote before using this.
    _fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
    if _fetch.returncode == 0:
        _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
        print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
    else:
        _git(["checkout", "-b", CKPT_BRANCH])
        print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh.")
    # keep the remote URL with the embedded token only as long as needed
    # for this session's pushes; drop the plaintext token variable now.
    del _token
else:
    print("checkpoint checkout already present this session -- reusing it.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """on_checkpoint_saved callback for run_training_loop (fl/round_loop.py,
    2026-11-16). Never raises -- a push failure costs a retry next round,
    not the training run; the checkpoint is already safe on local disk
    regardless of whether this push succeeds.

    2026-11-17 (FIX.md F17): each MVP run notebook now has its OWN
    dedicated CKPT_BRANCH, so this retry is defense-in-depth, not the
    primary fix -- it used to be the ONLY mitigation for 4 notebooks
    sharing one branch, which collided on every single push under real
    concurrent load (observed live). Kept in case this exact branch is
    ever reused by two sessions at once (e.g. the same notebook run
    twice): fetch+rebase onto the remote tip and retry once before
    giving up for this round (next round tries again regardless)."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            # Lost a race with a concurrent writer on this branch --
            # fetch + rebase onto the remote tip and retry once.
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# Fraction-grid cell: 10% malicious (5 of 50 clients),
# 5 rounds, A1 (model-replacement scaling). Same single-
# mechanism construction as every attack7 notebook's own run cell
# (notebooks/_build_mvp_run_notebooks.py's attack_run_cell) -- all
# 5 attackers submit the SAME crafted delta (a simplification
# noted there too, not independently-drawn attacks per client).
import numpy as np
import torch
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint
from attacks.a1_scaling import craft_scaled_update

REAL_ROUNDS = 5
CHEATER_CLIENT_IDX = list(range(5))

CKPT_PATH = f'{CKPT_DIR}/case1_tiny_cnn_fracgrid_10pct_r{REAL_ROUNDS}_nodp_f20.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed="bootstrap-tiny_cnn-case1-fracgrid-10pct".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

w_target = np.random.default_rng(CFG['seed'] + 99).normal(0, 1.0, size=dim)
malicious_delta = craft_scaled_update(flat_w.cpu().numpy().astype(np.float64), w_target)
print(f"[10pct] crafted malicious delta (clients {CHEATER_CLIENT_IDX}), raw norm:", np.linalg.norm(malicious_delta))

print("="*70)
print(f"[10pct] MALICIOUS CLIENTS: {CHEATER_CLIENT_IDX} ({len(CHEATER_CLIENT_IDX)} of {n_clients} = {100*len(CHEATER_CLIENT_IDX)/n_clients:.0f}%)")
print("="*70)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"[10pct] found checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

_eval_images, _eval_labels = case1_data['test']


def _eval_fn(round_idx, w_t_now):
    from fl.models import unflatten_to_state_dict
    device = next(model.parameters()).device
    model.load_state_dict(unflatten_to_state_dict(w_t_now.to(device), model_shapes))
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(_eval_images), 512):
            bi = _eval_images[i:i + 512].to(device)
            bl = _eval_labels[i:i + 512].to(device)
            correct += (model(bi).argmax(dim=1) == bl).sum().item()
    return {"acc": correct / len(_eval_images)}


final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case1_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    cheater_client_idx=CHEATER_CLIENT_IDX,
    cheater_force_delta=malicious_delta,
    on_checkpoint_saved=push_checkpoint_to_git,
    eval_fn=_eval_fn,
)
release_gpu_memory()

n_aborted = sum(1 for h in real_history if h.get('aborted'))
n_completed = len(real_history) - n_aborted
attacker_ever_accepted = {}
attacker_ever_rejected = {}
for h in real_history:
    if h.get("aborted"):
        print(f"[10pct]   round {h.get('round')}: ABORTED -- {h.get('reason')}")
        continue
    reasons = h.get("rejected_reasons", {})
    accepted_idxs = set(h.get("accepted_client_indices", []))
    rejected_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in reasons)
    accepted_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in accepted_idxs)
    for i in rejected_this_round:
        attacker_ever_rejected[i] = True
    for i in accepted_this_round:
        attacker_ever_accepted[i] = True
    print(f"[10pct]   round {h.get('round')}: accepted={h.get('accepted')}, "
          f"attackers rejected={rejected_this_round}, attackers accepted={accepted_this_round}, "
          f"aggregate_exact={h.get('aggregate_exact')}")

print(f"\n[10pct] {n_completed} of {len(real_history)} rounds completed, {n_aborted} aborted "
      f"(predicted abort rate at this fraction: see this notebook's title markdown).")
if attacker_ever_accepted:
    print(f"[10pct] FAIL: attacker(s) {sorted(attacker_ever_accepted)} had their scaled update accepted at least once.")
elif attacker_ever_rejected:
    print(f"[10pct] PASS: every eligible attacker ({sorted(attacker_ever_rejected)}) was rejected by V3 every time it submitted, across all {REAL_ROUNDS} rounds.")
else:
    print(f"[10pct] INCONCLUSIVE: no attacker was ever sampled/eligible in any round -- check sampling_rate/eligibility at this fraction.")

# Preserve this cell's results under a fraction-specific name before
# the next grid cell's RING_CELL/model_cell rebuild ring/model/flat_w
# (fresh per cell, same as the combo notebook's own convention) and
# before `real_history`/`n_aborted`/etc. get overwritten.
real_history_10pct = list(real_history)
n_aborted_10pct = n_aborted
n_completed_10pct = n_completed
attacker_ever_accepted_10pct = dict(attacker_ever_accepted)
attacker_ever_rejected_10pct = dict(attacker_ever_rejected)
CHEATER_CLIENT_IDX_10pct = list(CHEATER_CLIENT_IDX)


## Grid cell 2 of 3: 20% malicious (10 of 50 clients)

In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, verify_sub_share, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]

# Feldman-verify every sub-share before finalizing -- a real committee
# would discard a contribution that fails this and re-run DKG without
# the cheating holder; all contributions here are honest, so every
# check is expected to pass. Same check notebooks/traceability_demo
# already runs standalone; added here (2026-11-19) alongside pk_shares
# below so this notebook's committee is held to the same standard, now
# that §6.2 opening runs for real from this notebook's own evidence
# (not just the standalone demo).
for c in dkg_contributions:
    for j in key_holder_ids:
        assert verify_sub_share(c.sub_shares[j], c.feldman_commitments), (
            f"holder {c.holder_id}'s sub-share to {j} failed Feldman verification"
        )

PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
# pk_shares (2026-11-19): each holder's PUBLIC Feldman commitment --
# needed by crypto.elgamal_threshold.verify_partial inside
# fl/trace.py::run_opening_protocol to check a partial decryption
# before combining it. key_holder_shares (above) are the holders' own
# SECRET shares, needed to CREATE a partial in the first place
# (crypto.elgamal_threshold.compute_partial) -- both come out of the
# SAME DKG run, never recomputed independently.
pk_shares = {i: pk_share_for(i, dkg_contributions) for i in key_holder_ids}
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")


In [ ]:
# 7. Build the model.
# ~20.6K params, same model/dataset as LEFT.md item #1 and every attack7 notebook's base case.
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=10, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


## Checkpoint durability for this cell

Same git-backed checkpointing as the first cell -- this cell gets its own clone directory and its own dedicated branch, so it can never collide with another cell's checkpoint in this same session.

In [ ]:
# 7.5. Git-backed checkpoint durability (replaces Drive -- see markdown above).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints-fracgrid-1"
# 2026-11-17: each of the 4 MVP run notebooks gets its OWN branch, not
# one shared branch -- see FIX.md F17. Running all four at once on one
# shared branch caused every single push to collide and fail (12/12
# rounds, observed live) even with F16's fetch+rebase retry, because
# the retry itself can lose the SAME race under real concurrent load.
# A dedicated ref per notebook removes the race at the git level
# instead of mitigating it after the fact.
CKPT_BRANCH = "colab-checkpoints-case1-tiny_cnn-fracgrid-20pct"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    # does this notebook's own CKPT_BRANCH already exist on the remote
    # (pushed in an earlier, since-disconnected session)? if so, use it --
    # that's where the actual latest checkpoint lives. If not, this is new.
    # explicit destination refspec -- a plain "fetch origin <branch>"
    # only updates FETCH_HEAD, not refs/remotes/origin/<branch>, so the
    # checkout below would fail even though the fetch itself succeeded.
    # Verified against a real throwaway git remote before using this.
    _fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
    if _fetch.returncode == 0:
        _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
        print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
    else:
        _git(["checkout", "-b", CKPT_BRANCH])
        print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh.")
    # keep the remote URL with the embedded token only as long as needed
    # for this session's pushes; drop the plaintext token variable now.
    del _token
else:
    print("checkpoint checkout already present this session -- reusing it.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """on_checkpoint_saved callback for run_training_loop (fl/round_loop.py,
    2026-11-16). Never raises -- a push failure costs a retry next round,
    not the training run; the checkpoint is already safe on local disk
    regardless of whether this push succeeds.

    2026-11-17 (FIX.md F17): each MVP run notebook now has its OWN
    dedicated CKPT_BRANCH, so this retry is defense-in-depth, not the
    primary fix -- it used to be the ONLY mitigation for 4 notebooks
    sharing one branch, which collided on every single push under real
    concurrent load (observed live). Kept in case this exact branch is
    ever reused by two sessions at once (e.g. the same notebook run
    twice): fetch+rebase onto the remote tip and retry once before
    giving up for this round (next round tries again regardless)."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            # Lost a race with a concurrent writer on this branch --
            # fetch + rebase onto the remote tip and retry once.
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# Fraction-grid cell: 20% malicious (10 of 50 clients),
# 5 rounds, A1 (model-replacement scaling). Same single-
# mechanism construction as every attack7 notebook's own run cell
# (notebooks/_build_mvp_run_notebooks.py's attack_run_cell) -- all
# 10 attackers submit the SAME crafted delta (a simplification
# noted there too, not independently-drawn attacks per client).
import numpy as np
import torch
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint
from attacks.a1_scaling import craft_scaled_update

REAL_ROUNDS = 5
CHEATER_CLIENT_IDX = list(range(10))

CKPT_PATH = f'{CKPT_DIR}/case1_tiny_cnn_fracgrid_20pct_r{REAL_ROUNDS}_nodp_f20.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed="bootstrap-tiny_cnn-case1-fracgrid-20pct".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

w_target = np.random.default_rng(CFG['seed'] + 99).normal(0, 1.0, size=dim)
malicious_delta = craft_scaled_update(flat_w.cpu().numpy().astype(np.float64), w_target)
print(f"[20pct] crafted malicious delta (clients {CHEATER_CLIENT_IDX}), raw norm:", np.linalg.norm(malicious_delta))

print("="*70)
print(f"[20pct] MALICIOUS CLIENTS: {CHEATER_CLIENT_IDX} ({len(CHEATER_CLIENT_IDX)} of {n_clients} = {100*len(CHEATER_CLIENT_IDX)/n_clients:.0f}%)")
print("="*70)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"[20pct] found checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

_eval_images, _eval_labels = case1_data['test']


def _eval_fn(round_idx, w_t_now):
    from fl.models import unflatten_to_state_dict
    device = next(model.parameters()).device
    model.load_state_dict(unflatten_to_state_dict(w_t_now.to(device), model_shapes))
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(_eval_images), 512):
            bi = _eval_images[i:i + 512].to(device)
            bl = _eval_labels[i:i + 512].to(device)
            correct += (model(bi).argmax(dim=1) == bl).sum().item()
    return {"acc": correct / len(_eval_images)}


final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case1_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    cheater_client_idx=CHEATER_CLIENT_IDX,
    cheater_force_delta=malicious_delta,
    on_checkpoint_saved=push_checkpoint_to_git,
    eval_fn=_eval_fn,
)
release_gpu_memory()

n_aborted = sum(1 for h in real_history if h.get('aborted'))
n_completed = len(real_history) - n_aborted
attacker_ever_accepted = {}
attacker_ever_rejected = {}
for h in real_history:
    if h.get("aborted"):
        print(f"[20pct]   round {h.get('round')}: ABORTED -- {h.get('reason')}")
        continue
    reasons = h.get("rejected_reasons", {})
    accepted_idxs = set(h.get("accepted_client_indices", []))
    rejected_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in reasons)
    accepted_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in accepted_idxs)
    for i in rejected_this_round:
        attacker_ever_rejected[i] = True
    for i in accepted_this_round:
        attacker_ever_accepted[i] = True
    print(f"[20pct]   round {h.get('round')}: accepted={h.get('accepted')}, "
          f"attackers rejected={rejected_this_round}, attackers accepted={accepted_this_round}, "
          f"aggregate_exact={h.get('aggregate_exact')}")

print(f"\n[20pct] {n_completed} of {len(real_history)} rounds completed, {n_aborted} aborted "
      f"(predicted abort rate at this fraction: see this notebook's title markdown).")
if attacker_ever_accepted:
    print(f"[20pct] FAIL: attacker(s) {sorted(attacker_ever_accepted)} had their scaled update accepted at least once.")
elif attacker_ever_rejected:
    print(f"[20pct] PASS: every eligible attacker ({sorted(attacker_ever_rejected)}) was rejected by V3 every time it submitted, across all {REAL_ROUNDS} rounds.")
else:
    print(f"[20pct] INCONCLUSIVE: no attacker was ever sampled/eligible in any round -- check sampling_rate/eligibility at this fraction.")

# Preserve this cell's results under a fraction-specific name before
# the next grid cell's RING_CELL/model_cell rebuild ring/model/flat_w
# (fresh per cell, same as the combo notebook's own convention) and
# before `real_history`/`n_aborted`/etc. get overwritten.
real_history_20pct = list(real_history)
n_aborted_20pct = n_aborted
n_completed_20pct = n_completed
attacker_ever_accepted_20pct = dict(attacker_ever_accepted)
attacker_ever_rejected_20pct = dict(attacker_ever_rejected)
CHEATER_CLIENT_IDX_20pct = list(CHEATER_CLIENT_IDX)


## Grid cell 3 of 3: 30% malicious (15 of 50 clients)

In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, verify_sub_share, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]

# Feldman-verify every sub-share before finalizing -- a real committee
# would discard a contribution that fails this and re-run DKG without
# the cheating holder; all contributions here are honest, so every
# check is expected to pass. Same check notebooks/traceability_demo
# already runs standalone; added here (2026-11-19) alongside pk_shares
# below so this notebook's committee is held to the same standard, now
# that §6.2 opening runs for real from this notebook's own evidence
# (not just the standalone demo).
for c in dkg_contributions:
    for j in key_holder_ids:
        assert verify_sub_share(c.sub_shares[j], c.feldman_commitments), (
            f"holder {c.holder_id}'s sub-share to {j} failed Feldman verification"
        )

PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
# pk_shares (2026-11-19): each holder's PUBLIC Feldman commitment --
# needed by crypto.elgamal_threshold.verify_partial inside
# fl/trace.py::run_opening_protocol to check a partial decryption
# before combining it. key_holder_shares (above) are the holders' own
# SECRET shares, needed to CREATE a partial in the first place
# (crypto.elgamal_threshold.compute_partial) -- both come out of the
# SAME DKG run, never recomputed independently.
pk_shares = {i: pk_share_for(i, dkg_contributions) for i in key_holder_ids}
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")


In [ ]:
# 7. Build the model.
# ~20.6K params, same model/dataset as LEFT.md item #1 and every attack7 notebook's base case.
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=10, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


## Checkpoint durability for this cell

Same git-backed checkpointing as the first cell -- this cell gets its own clone directory and its own dedicated branch, so it can never collide with another cell's checkpoint in this same session.

In [ ]:
# 7.5. Git-backed checkpoint durability (replaces Drive -- see markdown above).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints-fracgrid-2"
# 2026-11-17: each of the 4 MVP run notebooks gets its OWN branch, not
# one shared branch -- see FIX.md F17. Running all four at once on one
# shared branch caused every single push to collide and fail (12/12
# rounds, observed live) even with F16's fetch+rebase retry, because
# the retry itself can lose the SAME race under real concurrent load.
# A dedicated ref per notebook removes the race at the git level
# instead of mitigating it after the fact.
CKPT_BRANCH = "colab-checkpoints-case1-tiny_cnn-fracgrid-30pct"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    # does this notebook's own CKPT_BRANCH already exist on the remote
    # (pushed in an earlier, since-disconnected session)? if so, use it --
    # that's where the actual latest checkpoint lives. If not, this is new.
    # explicit destination refspec -- a plain "fetch origin <branch>"
    # only updates FETCH_HEAD, not refs/remotes/origin/<branch>, so the
    # checkout below would fail even though the fetch itself succeeded.
    # Verified against a real throwaway git remote before using this.
    _fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
    if _fetch.returncode == 0:
        _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
        print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
    else:
        _git(["checkout", "-b", CKPT_BRANCH])
        print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh.")
    # keep the remote URL with the embedded token only as long as needed
    # for this session's pushes; drop the plaintext token variable now.
    del _token
else:
    print("checkpoint checkout already present this session -- reusing it.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """on_checkpoint_saved callback for run_training_loop (fl/round_loop.py,
    2026-11-16). Never raises -- a push failure costs a retry next round,
    not the training run; the checkpoint is already safe on local disk
    regardless of whether this push succeeds.

    2026-11-17 (FIX.md F17): each MVP run notebook now has its OWN
    dedicated CKPT_BRANCH, so this retry is defense-in-depth, not the
    primary fix -- it used to be the ONLY mitigation for 4 notebooks
    sharing one branch, which collided on every single push under real
    concurrent load (observed live). Kept in case this exact branch is
    ever reused by two sessions at once (e.g. the same notebook run
    twice): fetch+rebase onto the remote tip and retry once before
    giving up for this round (next round tries again regardless)."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            # Lost a race with a concurrent writer on this branch --
            # fetch + rebase onto the remote tip and retry once.
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# Fraction-grid cell: 30% malicious (15 of 50 clients),
# 5 rounds, A1 (model-replacement scaling). Same single-
# mechanism construction as every attack7 notebook's own run cell
# (notebooks/_build_mvp_run_notebooks.py's attack_run_cell) -- all
# 15 attackers submit the SAME crafted delta (a simplification
# noted there too, not independently-drawn attacks per client).
import numpy as np
import torch
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint
from attacks.a1_scaling import craft_scaled_update

REAL_ROUNDS = 5
CHEATER_CLIENT_IDX = list(range(15))

CKPT_PATH = f'{CKPT_DIR}/case1_tiny_cnn_fracgrid_30pct_r{REAL_ROUNDS}_nodp_f20.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed="bootstrap-tiny_cnn-case1-fracgrid-30pct".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

w_target = np.random.default_rng(CFG['seed'] + 99).normal(0, 1.0, size=dim)
malicious_delta = craft_scaled_update(flat_w.cpu().numpy().astype(np.float64), w_target)
print(f"[30pct] crafted malicious delta (clients {CHEATER_CLIENT_IDX}), raw norm:", np.linalg.norm(malicious_delta))

print("="*70)
print(f"[30pct] MALICIOUS CLIENTS: {CHEATER_CLIENT_IDX} ({len(CHEATER_CLIENT_IDX)} of {n_clients} = {100*len(CHEATER_CLIENT_IDX)/n_clients:.0f}%)")
print("="*70)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"[30pct] found checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

_eval_images, _eval_labels = case1_data['test']


def _eval_fn(round_idx, w_t_now):
    from fl.models import unflatten_to_state_dict
    device = next(model.parameters()).device
    model.load_state_dict(unflatten_to_state_dict(w_t_now.to(device), model_shapes))
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(_eval_images), 512):
            bi = _eval_images[i:i + 512].to(device)
            bl = _eval_labels[i:i + 512].to(device)
            correct += (model(bi).argmax(dim=1) == bl).sum().item()
    return {"acc": correct / len(_eval_images)}


final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case1_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    cheater_client_idx=CHEATER_CLIENT_IDX,
    cheater_force_delta=malicious_delta,
    on_checkpoint_saved=push_checkpoint_to_git,
    eval_fn=_eval_fn,
)
release_gpu_memory()

n_aborted = sum(1 for h in real_history if h.get('aborted'))
n_completed = len(real_history) - n_aborted
attacker_ever_accepted = {}
attacker_ever_rejected = {}
for h in real_history:
    if h.get("aborted"):
        print(f"[30pct]   round {h.get('round')}: ABORTED -- {h.get('reason')}")
        continue
    reasons = h.get("rejected_reasons", {})
    accepted_idxs = set(h.get("accepted_client_indices", []))
    rejected_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in reasons)
    accepted_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in accepted_idxs)
    for i in rejected_this_round:
        attacker_ever_rejected[i] = True
    for i in accepted_this_round:
        attacker_ever_accepted[i] = True
    print(f"[30pct]   round {h.get('round')}: accepted={h.get('accepted')}, "
          f"attackers rejected={rejected_this_round}, attackers accepted={accepted_this_round}, "
          f"aggregate_exact={h.get('aggregate_exact')}")

print(f"\n[30pct] {n_completed} of {len(real_history)} rounds completed, {n_aborted} aborted "
      f"(predicted abort rate at this fraction: see this notebook's title markdown).")
if attacker_ever_accepted:
    print(f"[30pct] FAIL: attacker(s) {sorted(attacker_ever_accepted)} had their scaled update accepted at least once.")
elif attacker_ever_rejected:
    print(f"[30pct] PASS: every eligible attacker ({sorted(attacker_ever_rejected)}) was rejected by V3 every time it submitted, across all {REAL_ROUNDS} rounds.")
else:
    print(f"[30pct] INCONCLUSIVE: no attacker was ever sampled/eligible in any round -- check sampling_rate/eligibility at this fraction.")

# Preserve this cell's results under a fraction-specific name before
# the next grid cell's RING_CELL/model_cell rebuild ring/model/flat_w
# (fresh per cell, same as the combo notebook's own convention) and
# before `real_history`/`n_aborted`/etc. get overwritten.
real_history_30pct = list(real_history)
n_aborted_30pct = n_aborted
n_completed_30pct = n_completed
attacker_ever_accepted_30pct = dict(attacker_ever_accepted)
attacker_ever_rejected_30pct = dict(attacker_ever_rejected)
CHEATER_CLIENT_IDX_30pct = list(CHEATER_CLIENT_IDX)


In [ ]:
print("\n" + "="*70)
print("A1 FRACTION-SWEEP SUMMARY -- defense-trigger-rate vs. malicious fraction")
print("="*70)
print(f"{'fraction':>10} | {'rounds':>7} | {'completed':>10} | {'aborted':>8} | {'attackers rejected':>20} | {'attackers accepted':>20}")
print(f"{'10%':>10} | {len(real_history_10pct):>7} | {n_completed_10pct:>10} | {n_aborted_10pct:>8} | {len(attacker_ever_rejected_10pct):>20} | {len(attacker_ever_accepted_10pct):>20}")
print(f"{'20%':>10} | {len(real_history_20pct):>7} | {n_completed_20pct:>10} | {n_aborted_20pct:>8} | {len(attacker_ever_rejected_20pct):>20} | {len(attacker_ever_accepted_20pct):>20}")
print(f"{'30%':>10} | {len(real_history_30pct):>7} | {n_completed_30pct:>10} | {n_aborted_30pct:>8} | {len(attacker_ever_rejected_30pct):>20} | {len(attacker_ever_accepted_30pct):>20}")
